# 07 · Normalisation

`s.normalize(method, **settings)`. Removes intensity differences that do not come from the sample (laser power, focus, integration time), so spectra can be compared. Usually done **after** despike and baseline.

| method | what it does | use when |
|---|---|---|
| `snv` (default) | subtract mean, divide by standard deviation | general purpose; works on one spectrum |
| `max`, `minmax` | divide by the maximum / rescale 0..1 | quick look; one noisy point decides |
| `area`, `vector` | divide by area / L2 norm | baseline already removed |
| `rsnv` | SNV with median and MAD | a few huge bands dominate |
| `band` | divide by one band's height or area (`region=`) | you have an internal standard |
| `msc`, `emsc` | fit each spectrum to the mean (+ polynomial) | a set of spectra; `emsc` also removes sloping backgrounds |
| `pqn` | median of ratios to the median spectrum | some bands really change (concentration series) |

Any one-factor method takes `region=(lo, hi)` to compute the factor from that range only. How they compare: `docs/benchmarks.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import spectrum_toolkit as stk
from spectrum_toolkit.normalize import METHODS, DEFAULTS

cv = stk.load('../data/CV10-4M.xlsx', verbose=False).despike(verbose=False).baseline()

def spread(Y):
    '''How different the repeats still are, in % of the signal variation (lower = more alike).'''
    return 100 * np.sqrt(np.mean(Y.var(axis=0))) / np.std(Y.mean(axis=0))

## 1. The 14 CV repeats before and after

In [ ]:
methods = ['max', 'area', 'snv', 'emsc', 'pqn']
fig, ax = plt.subplots(1, len(methods) + 1, figsize=(20, 3.5))
ax[0].plot(cv.x, cv.Y.T, lw=0.5); ax[0].set_title(f'baseline only: spread {spread(cv.Y):.0f} %')
for a, m in zip(ax[1:], methods):
    n = cv.normalize(m)
    a.plot(n.x, n.Y.T, lw=0.5); a.set_title(f'{m}: spread {spread(n.Y):.0f} %')
plt.tight_layout(); plt.show()

## 2. Play with the settings

In [ ]:
from ipywidgets import interact, Dropdown, FloatRangeSlider, IntSlider, Checkbox

def show(method='snv', use_region=False, region=(1580, 1640), measure='height', order=2):
    given = dict(region=tuple(region) if use_region or method == 'band' else None, measure=measure, order=order)
    settings = {k: v for k, v in given.items() if k in DEFAULTS[method]}
    try:
        n = cv.normalize(method, **settings)
    except ValueError as e:
        print(e); return
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(n.x, n.Y.T, lw=0.6)
    if 'region' in settings and settings['region']:
        ax.axvspan(*settings['region'], color='y', alpha=0.2)
    ax.set_title(f'{method} {settings}: spread {spread(n.Y):.1f} %'); plt.show()

interact(show, method=Dropdown(options=METHODS),
         region=FloatRangeSlider(value=(1580, 1640), min=float(cv.x.min()), max=float(cv.x.max()), step=5),
         measure=Dropdown(options=['height', 'area']), order=IntSlider(value=2, min=0, max=5));

`band` always uses the region (pick a band that should be the same in every spectrum). For the others, tick `use_region` to compute the factor from the region only.

## 3. One spectrum against a reference (msc / emsc / pqn)

In [ ]:
ref = cv.Y.mean(axis=0)
one = cv[3].normalize('msc', reference=ref)
one.history[-1]